In [3]:
from kan import KAN, LBFGS, MLP
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from tqdm import tqdm
import json

device = torch.device('cpu')
print(device)

cpu


In [4]:
def sol_fun(x):
    half = x.shape[1] // 2
    m = x[:, :half]
    v = x[:, half:2*half]
    return torch.sum(0.5 * m * (v ** 2), dim=1, keepdim=True)

In [5]:
ranges = [-1, 1]
n_train = 1000
n_test = 200
steps = 100
grid = 5
k=3
lr=0.1

In [ ]:
def generate_dataset(dim, n_train=None, n_test=None):
    if n_train is None:
        n_train = globals().get('n_train', 1000)
    if n_test is None:
        n_test = globals().get('n_test', 200)
    x_train = torch.rand((n_train, dim), device=device) * (ranges[1] - ranges[0]) + ranges[0]
    y_train = sol_fun(x_train)
    x_test = torch.rand((n_test, dim), device=device) * (ranges[1] - ranges[0]) + ranges[0]
    y_test = sol_fun(x_test)
    dataset = {
        'train_input': x_train,
        'train_label': y_train,
        'test_input': x_test,
        'test_label': y_test
    }
    return dataset

In [11]:
def train_kan(width, dataset):
    criterion = torch.nn.MSELoss()
    rmses = []
    r2s = []
    preds = []
    model = KAN(width=width, grid=grid, k=k, seed=1, device=device)
    model = model.speed()
    num_params = sum(1 for _ in model.parameters())

    optimizer = LBFGS(
            model.parameters(), 
            lr=lr, 
            history_size=10, 
            line_search_fn="strong_wolfe",
            tolerance_grad=1e-32, 
            tolerance_change=1e-32, 
            tolerance_ys=1e-32
        )
    pbar = tqdm(range(steps), desc='description', ncols=100)

    for _ in pbar:
        def closure():
            optimizer.zero_grad()
            loss = criterion(model(dataset['train_input']), dataset['train_label'])
            loss.backward()
            return loss

        if _ % (steps//10)== 0 and _ < steps // 2:
            model.update_grid_from_samples(dataset['train_input'])

        optimizer.step(closure)
        with torch.no_grad():
            test_pred = model(dataset['test_input'])
            mse = criterion(test_pred, dataset['test_label'])
            rmse = torch.sqrt(mse).item()
            r2 = r2_score(dataset['test_label'].cpu().numpy(), test_pred.cpu().numpy())

        pbar.set_description("rmse: %.2e | r2: %.2e " % (rmse, r2))

        rmses.append(rmse)
        r2s.append(r2)
        preds.append(test_pred)
            
    results = {
        "rmses": rmses,
        "r2s": r2s,
        "preds": preds,
        "param_counts": num_params
    }
    return results

In [8]:
def plot_results(rmses, r2s):
    _, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    ax1.plot(r2s, marker='o')
    ax1.set_xlabel('steps')
    ax1.legend(['R2 score'])

    ax2.plot(rmses, marker='o')
    ax2.set_yscale('log')
    ax2.set_xlabel('steps')
    ax2.legend(['RMSE'])

    plt.tight_layout()
    print("Final RMSE: %.2e" % rmses[-1])

In [9]:
def train_mlp(width, dataset):
    
    def r2():
        with torch.no_grad():
            pred = mlp_model(dataset['test_input'].to(mlp_model.device))
        target = dataset['test_label'].to(mlp_model.device)
        ss_res = torch.sum((target - pred) ** 2)
        ss_tot = torch.sum((target - torch.mean(target)) ** 2)
        return 1 - (ss_res / ss_tot)
    
    mlp_model = MLP.MLP(width=width, act='silu', device='cpu')
    mlp_params = sum(p.numel() for p in mlp_model.parameters() if p.requires_grad)
    results = mlp_model.fit(dataset, opt="LBFGS", steps=100, lr=0.1, metrics=[r2])
    results['param_counts'] = mlp_params
    return results

In [ ]:
kan_results = {}
mlp_results = {}
dim = 1000
target_r2 = 0.95
n_train = 1000
max_n_train = 100000
while n_train <= max_n_train:
    print(f"Training with dim={dim}, n_train={n_train}")
    dataset = generate_dataset(dim=dim, n_train=n_train)
    kan_results[n_train] = train_kan(width=[dim, dim, 1], dataset=dataset)
    kan_param = kan_results[n_train]['param_counts']
    mlp_hidden_width = (kan_param - 1) // 4
    mlp_results[n_train] = train_mlp(width=[dim, mlp_hidden_width, 1], dataset=dataset)
    mlp_param = mlp_results[n_train]['param_counts']
    final_r2 = kan_results[n_train]['r2s'][-1]
    print(f"Final R2 for N={n_train}: {final_r2:.4f}")
    if final_r2 >= target_r2:
        print(f"Reached target R2={final_r2:.4f} with N={n_train}")
        break
    n_train *= 2
else:
    print(f"Did not reach target R2={target_r2} by N={max_n_train}")

NameError: name 'generate_dataset' is not defined

In [ ]:
for kan_result in kan_results:
    plot_results(kan_result['rmses'], kan_result['r2s'])

In [ ]:
for mlp_result in mlp_results:
    plot_results(mlp_result['rmses'], mlp_result['r2s'])

In [ ]:
with open('kan_results.json', 'w') as json_file:
    json.dump(kan_results, json_file, indent=4)
with open('mlp_results.json', 'w') as json_file:
    json.dump(mlp_results, json_file, indent=4)